In [ ]:
############ IMPORT LIBRARIES ############

from QNBAnalytics_ML import data
from QNBAnalytics_ML import skills_api
import pandas as pd
import numpy as np
import pickle
import datetime

version = "scoring_policy_adjustment"
start_time = datetime.datetime.now()

In [ ]:
############ ENTER PARAMETERS ############

index_col = "ID"  # index column
target_col = "TARGET"  # target column
cols_to_drop=[]

In [ ]:
############ READ TEST DATA ############

database_username = pd.read_table('Data/user', header = None)[0][0]  # enter your db username
database_password = pd.read_table('Data/pass', header = None)[0][0]  # enter your database password

engine=data.connect_to_sql(database_username, database_password)
with open(f'Data/test_data_sql_{version}.txt') as f:  test_data_sql = f.read()   # enter test data path
test = data.data_load(engine, sql = test_data_sql)
test = test.set_index(index_col)

x_test = test
y_test = []

In [ ]:
############ READ SCORING DATA FROM CSV/XLSX ############

# Option 1: Read from CSV
new_data = pd.read_csv('Data/score_data.csv')

# Option 2: Read from Excel (uncomment if using XLSX)
# new_data = pd.read_excel('Data/score_data.xlsx')

new_data = new_data.set_index(index_col)

# Case 1: If you have TARGET column (for validation)
if 'TARGET' in new_data.columns:
    new_data = new_data.rename(columns={target_col: "TARGET"})
    y_new = new_data["TARGET"]
    x_new = new_data.drop(columns=["TARGET"]+cols_to_drop, inplace=False)
    print(f"Scoring data with target: {x_new.shape}")
# Case 2: No TARGET column (real production scoring)
else:
    x_new = new_data.drop(columns=cols_to_drop, errors='ignore', inplace=False)
    y_new = None
    print(f"Scoring data (no target): {x_new.shape}")

In [ ]:
############ CREATE PIPELINE ############

class Pipeline:
    def apply(self, x_test, y_test=[]):
        predicts = self.pipeline.test(test = x_test, y_test=y_test)
        return predicts

In [ ]:
############ LOAD MODEL ############
model_version = "training"
pipeline_base = Pipeline()
pipeline_not_good = Pipeline()
pipeline_good = Pipeline()
pipeline_meta = Pipeline()

model_path = f'Models/base_model_{model_version}.pkl'
with open(model_path, 'rb') as f:
    pipeline_base.pipeline = pickle.load(f)
    
model_path = f'Models/not_good_model_{model_version}.pkl'
with open(model_path, 'rb') as f:
    pipeline_not_good.pipeline = pickle.load(f)

model_path = f'Models/good_model_{model_version}.pkl'
with open(model_path, 'rb') as f:
    pipeline_good.pipeline = pickle.load(f)

model_path = f'Models/meta_model_{model_version}.pkl'
with open(model_path, 'rb') as f:
    pipeline_meta.pipeline = pickle.load(f)


In [ ]:
############ SELECTED MODELS ############

not_good_selected_model = 'Logistic Regression'
good_selected_model = 'LGBM'

not_good_model = 'logistic_regression'
good_model = 'lgbm'

In [ ]:
############ GENERATE FEATURES FOR LAYER_3 ############

base_test_proba_dict = pipeline_base.apply(x_test, y_test)
base_test_proba = base_test_proba_dict["".join(key for key in base_test_proba_dict if "Logistic Regression" in key)][0]

not_good_test_proba_dict = pipeline_not_good.apply(x_test, y_test)
not_good_test_proba = not_good_test_proba_dict["".join(key for key in not_good_test_proba_dict if f"{not_good_selected_model}" in key)][0]
good_test_proba_dict = pipeline_good.apply(x_test, y_test)
good_test_proba = good_test_proba_dict["".join(key for key in good_test_proba_dict if f"{good_selected_model}" in key)][0]

base_test_proba = pd.DataFrame(base_test_proba, index = x_test.index)
good_test_proba = pd.DataFrame(good_test_proba, index = x_test.index)
not_good_test_proba = pd.DataFrame(not_good_test_proba, index = x_test.index)

x_test_meta = pd.concat([base_test_proba,good_test_proba,not_good_test_proba], axis = 1)
x_test_meta.columns = ["Base","Good","Not_Good"]
y_test_meta = y_test

In [ ]:
############ ADVANCED PIPELINE RESULTS ############

meta_test_proba_dict = pipeline_meta.apply(x_test_meta, y_test_meta)
meta_test_proba = meta_test_proba_dict["".join(key for key in meta_test_proba_dict if "Logistic Regression" in key)][0]


In [ ]:
############ CALCULATE CREDIT SCORES ############

ref=200
odds_at_ref=100
points_to_double=20


default_rate = meta_test_proba
default_rate = np.where(default_rate == 0, 0.00001, default_rate)
odds = (1/default_rate)-1
meta_test_score = ((np.log(odds)-np.log(odds_at_ref))/np.log(2)) * (points_to_double) + ref


meta_test_proba = pd.DataFrame(meta_test_proba, index = x_test_meta.index)
meta_test_score = pd.DataFrame(meta_test_score, index = x_test_meta.index)

test_scores = pd.concat([meta_test_proba, meta_test_score], axis = 1)
test_scores.columns = ["PROBABILITY","SCORE"]

print(test_scores)
print("Runtime: %s" % (datetime.datetime.now()- start_time))

### Policy Adjustment

In [ ]:
def compute_policy_adjusted_score(score, policy_constant, policy_multiplier):
    discounted_score = score * policy_multiplier
    policy_adjusted_score = min(score, policy_constant, discounted_score)
    return policy_adjusted_score

In [ ]:
score_df = pd.merge(test[["SCR_CONSTANT","SCR_MULTIPLIER"]], test_scores, left_index=True, right_index=True).sort_index()
score_df["POLICY_ADJUSTED_SCORE"] = [compute_policy_adjusted_score(a,b,c) for _, (a,b,c) in score_df[["SCORE","SCR_CONSTANT","SCR_MULTIPLIER"]].iterrows()]
score_df = score_df.rename({"SCORE":"MODEL_SCORE"}, axis=1)
# score_df.sort_index().to_excel(f"Output/SCORES_POLICY_ADJUSTED.xlsx")
print(score_df)